# Conditional D3PM for 7 × 7 Queens

Generate a **binary queen-occupancy map** from discrete noise, conditioned on
the original region coloring. The generated state contains only **0 = no queen**
and **1 = queen**. The coloring is a separate, unchanged input at every step.

Exactly one queen belongs to every row, column and region; queens cannot touch
diagonally. Longer diagonals are allowed.

1. The same prepared puzzles, split and augmentations as the DFM experiment.
2. A discrete forward noising process and a conditional U-Net.
3. Online D3PM training, checkpoint saving and loading.
4. Reverse diffusion trajectories from independent binary noise.
5. Held-out solution rates, retry curves and a solvability study.

**Matched budget.** Like the neighboring DFM notebook: 100 epochs × 4,096
augmented examples, batch size 64 (**6,400 optimizer steps**), one network
evaluation per training batch, **100 network calls per generated attempt**
and **32 attempts per evaluated puzzle**. U-Net widths, time embedding,
optimizer and evaluation-puzzle selection also match. Input/output heads
differ (11 → 2 channels here, 17 → 8 in DFM), so parameter counts and FLOPs
are close rather than identical. Loss magnitudes are different objectives;
compare held-out solution rates under the same budget.

Reference: [Austin et al., Structured Denoising Diffusion Models in Discrete
State-Spaces, §§2–3](https://arxiv.org/html/2107.03006).


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").exists()
)
sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))
import html
from collections import Counter, defaultdict
import json
import ipywidgets as widgets
import numpy as np
import plotly.graph_objects as go
import torch
from IPython.display import HTML, display
from plotly.subplots import make_subplots
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset, RandomSampler
from _shared.model import Denoiser
from _shared.queens_data import (
    BOARD_SIZE,
    EMPTY,
    load_queens_dataset,
    solution_checks,
    split_puzzles,
)

SEED = 17
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 2
DISPLAY_CLASSES = BOARD_SIZE + 1
DIFFUSION_STEPS = 100
GENERATION_STEPS = DIFFUSION_STEPS
ARTIFACTS = PROJECT_ROOT / "artifacts" / "queens7"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = ARTIFACTS / "queens7_d3pm.pt"
PALETTE = [
    "#B8A5E8",
    "#F0B48B",
    "#92C9E8",
    "#A9D4A4",
    "#E8CD83",
    "#ECA6BB",
    "#9FB9D4",
    "#F1F3F5",
]
CATEGORICAL_SCALE = [
    [position, color]
    for index, color in enumerate(PALETTE)
    for position in (index / DISPLAY_CLASSES, (index + 1) / DISPLAY_CLASSES)
]
PLOT_CONFIG = {"responsive": True, "displaylogo": False, "displayModeBar": False}
display(
    HTML(
        f"<b>Queens 7×7 · Conditional binary D3PM</b><br>"
        f"Device: {html.escape(str(DEVICE))} · state: 0/1 · {GENERATION_STEPS} reverse steps"
    )
)

## 1. Prepared dataset

The collection combines already solved **AMAZE**, **Queens.game** and **Queendom** examples with synthetic pairs prepared offline. Every record includes its original source and queen coordinates.

- External answers are copied from their sources; their uniqueness claim is trusted.
- Synthetic boards were checked for uniqueness once during offline preparation. Synthetic regions are grown round robin, then their boundaries are refined to obtain a unique solution while preserving connectivity and 2–12 cells per region.
- Rotated, reflected and color-renamed duplicates are removed before splitting.
- Only training uses geometric augmentations and color permutations.
- We make our own split; this is not an official AMAZE benchmark.

Sources: [AMAZE data](https://huggingface.co/datasets/piekenius123/Amaze), [AMAZE construction](https://github.com/spatigen/amaze/tree/main/queen-generator), [Queens.game](https://queens.game/about.html), [Queendom](https://github.com/bonesmoses/queendom).

To rebuild outside the notebook, run <code>scripts/collect_queens7.py</code>, then <code>scripts/generate_queens7.py --count 5000</code>.


In [2]:
puzzles = load_queens_dataset(PROJECT_ROOT / "data" / "queens7")
splits = split_puzzles(puzzles["fingerprints"], seed=SEED)
regions, queen_masks = puzzles["regions"], puzzles["queens"]
target_tokens = np.where(queen_masks, regions, EMPTY).astype(np.int64)


class QueensDataset(Dataset):
    def __init__(self, indices, augment=False):
        self.indices = np.asarray(indices, dtype=np.int64)
        self.augment = augment

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        source_index = int(self.indices[index])
        board = torch.from_numpy(regions[source_index].astype(np.int64))
        queens = torch.from_numpy(queen_masks[source_index].copy())
        if self.augment:
            turns = int(torch.randint(4, ()).item())
            board = torch.rot90(board, turns, dims=(-2, -1))
            queens = torch.rot90(queens, turns, dims=(-2, -1))
            if bool(torch.randint(2, ()).item()):
                board, queens = board.flip(-1), queens.flip(-1)
            board = torch.randperm(BOARD_SIZE)[board]
        return board.contiguous(), queens.long().contiguous()


train_dataset = QueensDataset(splits["train"], augment=True)
validation_dataset = QueensDataset(splits["validation"])
test_dataset = QueensDataset(splits["test"])
source_summary = " · ".join(
    f"{html.escape(name)}: {count:,}"
    for name, count in puzzles["metadata"]["source_counts"].items()
)
split_summary = " · ".join(
    f"{name}: {len(indices):,}" for name, indices in splits.items()
)
display(
    HTML(
        f"<b>{len(regions):,} distinct puzzles</b><br>{source_summary}<br>{split_summary}"
    )
)

In [3]:
def add_board(fig, tokens, board, row, col, queens=False):
    tokens, board = np.asarray(tokens), np.asarray(board)
    fig.add_trace(
        go.Heatmap(
            z=tokens,
            zmin=-0.5,
            zmax=DISPLAY_CLASSES - 0.5,
            colorscale=CATEGORICAL_SCALE,
            showscale=False,
            xgap=1,
            ygap=1,
            customdata=board,
            hovertemplate="row=%{y}, col=%{x}<br>region=%{customdata}<br>token=%{z}<extra></extra>",
        ),
        row=row,
        col=col,
    )
    xs, ys = [], []
    for r in range(BOARD_SIZE):
        for c in range(BOARD_SIZE):
            if c + 1 < BOARD_SIZE and board[r, c] != board[r, c + 1]:
                xs.extend([c + 0.5, c + 0.5, None])
                ys.extend([r - 0.5, r + 0.5, None])
            if r + 1 < BOARD_SIZE and board[r, c] != board[r + 1, c]:
                xs.extend([c - 0.5, c + 0.5, None])
                ys.extend([r + 0.5, r + 0.5, None])
    fig.add_trace(
        go.Scatter(
            x=xs,
            y=ys,
            mode="lines",
            line=dict(color="#64748B", width=1.5),
            hoverinfo="skip",
            showlegend=False,
        ),
        row=row,
        col=col,
    )
    if queens:
        qr, qc = np.where(tokens != EMPTY)
        fig.add_trace(
            go.Scatter(
                x=qc,
                y=qr,
                mode="text",
                text=["♛"] * len(qr),
                textfont=dict(size=25, color="#172033"),
                hoverinfo="skip",
                showlegend=False,
            ),
            row=row,
            col=col,
        )
    fig.update_xaxes(
        range=[-0.5, BOARD_SIZE - 0.5],
        showgrid=False,
        zeroline=False,
        showticklabels=False,
        constrain="domain",
        fixedrange=True,
        row=row,
        col=col,
    )
    xaxis_name = fig.get_subplot(row, col).xaxis.plotly_name.replace("axis", "")
    fig.update_yaxes(
        range=[BOARD_SIZE - 0.5, -0.5],
        showgrid=False,
        zeroline=False,
        showticklabels=False,
        scaleanchor=xaxis_name,
        scaleratio=1,
        constrain="domain",
        fixedrange=True,
        row=row,
        col=col,
    )


def style_boards(fig, title, rows=1):
    fig.update_layout(
        title=dict(text=title, x=0, font=dict(size=22)),
        template="plotly_white",
        autosize=True,
        height=rows * 225 + 90,
        margin=dict(l=10, r=10, t=85, b=20),
        font=dict(family="Arial, sans-serif", size=15),
        showlegend=False,
    )
    fig.update_annotations(font=dict(size=14))
    return fig


def mask_to_tokens(mask, coloring):
    # A display/scoring adapter only: diffusion states always remain binary.
    return np.where(np.asarray(mask, dtype=bool), np.asarray(coloring), EMPTY)


def add_mask(fig, mask, coloring, row, col, queens=False):
    mask = np.asarray(mask, dtype=np.int64)
    heatmap_index = len(fig.data)
    add_board(fig, mask_to_tokens(mask, coloring), coloring, row, col, queens=queens)
    heatmap = fig.data[heatmap_index]
    heatmap.z = mask
    heatmap.zmin, heatmap.zmax = -0.5, 1.5
    heatmap.colorscale = [
        [0.0, "#172033"],
        [0.5, "#172033"],
        [0.5, "#F8FAFC"],
        [1.0, "#F8FAFC"],
    ]
    heatmap.hovertemplate = (
        "row=%{y}, col=%{x}<br>region=%{customdata}" "<br>queen=%{z}<extra></extra>"
    )

### 1.1 Examples from every source

Draw up to two distinct examples per source before augmentation, with the supplied queens directly on the complete region coloring. Queendom contributes one example.


In [4]:
SOURCE_EXAMPLES = 2
source_example_rng = np.random.default_rng(SEED + 201)
dataset_source_examples = {}
for source_name in sorted(set(puzzles["source_names"].tolist())):
    available = np.flatnonzero(puzzles["source_names"] == source_name)
    chosen = source_example_rng.choice(
        available,
        size=min(SOURCE_EXAMPLES, len(available)),
        replace=False,
    )
    dataset_source_examples[source_name] = chosen
    fig = make_subplots(
        rows=1,
        cols=len(chosen),
        subplot_titles=[f"Example {number}" for number in range(1, len(chosen) + 1)],
        horizontal_spacing=0.07,
    )
    for col, index in enumerate(chosen, start=1):
        add_board(fig, regions[index], regions[index], 1, col)
        queen_rows, queen_cols = np.where(queen_masks[index])
        fig.add_trace(
            go.Scatter(
                x=queen_cols,
                y=queen_rows,
                mode="text",
                text=[chr(0x265B)] * len(queen_rows),
                textfont=dict(size=25, color="#172033"),
                hoverinfo="skip",
                showlegend=False,
            ),
            row=1,
            col=col,
        )
    style_boards(
        fig,
        f"{source_name} ? {len(available):,} puzzles",
        rows=1,
    ).show(config=PLOT_CONFIG)

### 1.2 Number of colorings per solution

Group the entire prepared dataset by the exact queen positions: the queen's column in each row.
Let $K(s)$ be the number of distinct region maps assigned to solution $s$. Under a uniform choice
among the distinct observed solutions,

$$
P(K=k)=\frac{\#\{s:K(s)=k\}}{\#\{\text{distinct observed solutions}\}}.
$$

This is the empirical distribution in our dataset, with no augmentation. The dataset has already
removed duplicate region maps up to rotations, reflections and color relabeling.
Here queen positions retain their exact orientation; rotated queen placements count as different solutions.


In [5]:
colorings_by_solution = defaultdict(list)
for index, record in enumerate(puzzles["records"]):
    queen_columns = tuple(column for row, column in sorted(record["queens"]))
    colorings_by_solution[queen_columns].append(index)

coloring_multiplicities = np.asarray(
    [len(indices) for indices in colorings_by_solution.values()],
    dtype=np.int64,
)
multiplicity_counts = Counter(coloring_multiplicities.tolist())
multiplicity_support = np.arange(1, int(coloring_multiplicities.max()) + 1)
multiplicity_frequencies = np.asarray(
    [multiplicity_counts.get(int(count), 0) for count in multiplicity_support]
)
multiplicity_probabilities = multiplicity_frequencies / len(colorings_by_solution)
mean_colorings = float(coloring_multiplicities.mean())
median_colorings = float(np.median(coloring_multiplicities))

fig = go.Figure(
    go.Bar(
        x=multiplicity_support,
        y=multiplicity_probabilities,
        customdata=multiplicity_frequencies,
        marker_color="#5267DB",
        hovertemplate=(
            "Colorings per solution: %{x}<br>"
            "Distinct solutions: %{customdata}<br>"
            "Probability: %{y:.2%}<extra></extra>"
        ),
    )
)
fig.add_vline(
    x=mean_colorings,
    line_color="#DC6557",
    line_width=2,
    line_dash="dash",
    annotation_text=f"Mean = {mean_colorings:.2f}",
    annotation_position="top right",
    annotation_font_color="#B85045",
)
fig.update_layout(
    title=dict(
        text=f"Colorings per solution · {len(regions):,} puzzles · "
        f"{len(colorings_by_solution):,} distinct queen placements",
        x=0,
    ),
    template="plotly_white",
    autosize=True,
    height=440,
    xaxis_title="Number of distinct colorings per solution",
    yaxis_title="Probability · uniformly selected distinct solution",
    font=dict(family="Arial, sans-serif", size=15),
    margin=dict(l=70, r=25, t=85, b=65),
    bargap=0.18,
)
fig.update_xaxes(
    dtick=max(1, int(np.ceil(multiplicity_support[-1] / 30))),
    range=[0.3, float(multiplicity_support[-1]) + 0.7],
)
fig.update_yaxes(tickformat=".0%", rangemode="tozero")
fig.show(config=PLOT_CONFIG)
display(
    HTML(
        f"<b>{len(colorings_by_solution):,} unique queen placements</b> · "
        f"mean {mean_colorings:.2f} colorings · median {median_colorings:g} · "
        f"range {coloring_multiplicities.min()}–{coloring_multiplicities.max()}<br>"
        f"More than one coloring: {np.mean(coloring_multiplicities > 1):.2%} of solutions."
    )
)

## 2. Binary forward diffusion

Write the clean queen mask as $x_0 \in \{0,1\}^{7\times7}$ and the fixed
region map as $c$. **Only $x$ is noised; $c$ is never corrupted.**

Each forward step independently keeps a cell or replaces it with a fair bit:

$$
Q_t = \alpha_t I + (1-\alpha_t)U,\qquad
U=\tfrac12\mathbf{1}\mathbf{1}^{\mathsf T}.
$$

For $\bar\alpha_t=\prod_{s=1}^t\alpha_s$, the cumulative transition is

$$
\bar Q_t=\bar\alpha_t I+(1-\bar\alpha_t)U,\qquad
q(x_t^i\mid x_0^i)=\bar Q_t[x_0^i,x_t^i].
$$

Use a cosine schedule with $T=100$ and offset $s=0.008$:

$$
\bar\alpha_t =
\frac{\cos^2\!\left(\frac{t/T+s}{1+s}\frac{\pi}{2}\right)}
     {\cos^2\!\left(\frac{s}{1+s}\frac{\pi}{2}\right)},\qquad
\bar\alpha_0=1,\quad \bar\alpha_T=0.
$$

The terminal state is **exactly independent Bernoulli(1/2) noise**.
Training draws any $x_t$ directly from $\bar Q_t$; the visualization below
follows the actual Markov chain using successive $Q_t$ transitions.
Dark cells are 0 and light cells are 1. The first panel shows the unchanged
coloring supplied to the network.


In [6]:
DIFFUSION_CONFIG = dict(steps=DIFFUSION_STEPS, cosine_offset=0.008)


def make_diffusion(config, device):
    steps = int(config["steps"])
    offset = float(config["cosine_offset"])
    if steps < 1 or offset < 0:
        raise ValueError("Use a positive step count and a nonnegative cosine offset.")
    times = torch.linspace(0, 1, steps + 1, dtype=torch.float64)
    alpha_bar = torch.cos((times + offset) / (1 + offset) * torch.pi / 2).square()
    alpha_bar /= alpha_bar[0].clone()
    alpha_bar[0], alpha_bar[-1] = 1.0, 0.0
    retention = torch.ones_like(alpha_bar)
    retention[1:] = alpha_bar[1:] / alpha_bar[:-1]
    identity = torch.eye(NUM_CLASSES, dtype=torch.float64)
    uniform = torch.full_like(identity, 1.0 / NUM_CLASSES)
    transitions = (
        retention[:, None, None] * identity + (1 - retention[:, None, None]) * uniform
    )
    cumulative = (
        alpha_bar[:, None, None] * identity + (1 - alpha_bar[:, None, None]) * uniform
    )
    # Index order: [t, observed x_t, possible clean x_0, previous x_{t-1}].
    joint = torch.zeros(steps + 1, 2, 2, 2, dtype=torch.float64)
    for time in range(1, steps + 1):
        joint[time] = torch.einsum(
            "kj,ak->jak", transitions[time], cumulative[time - 1]
        )
    posterior = joint / joint.sum(dim=-1, keepdim=True).clamp_min(1e-30)
    return {
        "alpha_bar": alpha_bar.to(device=device, dtype=torch.float32),
        "retention": retention.to(device=device, dtype=torch.float32),
        "joint_log": joint.log().to(device=device, dtype=torch.float32),
        "posterior": posterior.to(device=device, dtype=torch.float32),
    }


diffusion = make_diffusion(DIFFUSION_CONFIG, DEVICE)


def q_sample(clean, timesteps, generator=None):
    alpha_bar = diffusion["alpha_bar"].to(clean.device)[timesteps]
    flip_probability = (1 - alpha_bar[:, None, None]) / 2
    flips = (
        torch.rand(clean.shape, device=clean.device, generator=generator)
        < flip_probability
    )
    return clean.long() ^ flips.long()


FORWARD_TIMESTEPS = tuple(
    sorted(set(np.rint(np.linspace(0, DIFFUSION_STEPS, 6)).astype(int)))
)
forward_index = int(splits["validation"][0])
forward_coloring = regions[forward_index]
forward_state = torch.as_tensor(
    queen_masks[forward_index], dtype=torch.long, device=DEVICE
)
forward_generator = torch.Generator(device=DEVICE).manual_seed(SEED + 101)
forward_snapshots = {0: forward_state.cpu().numpy().copy()}
for time in range(1, DIFFUSION_STEPS + 1):
    flip_probability = (1 - diffusion["retention"][time]) / 2
    flips = (
        torch.rand(forward_state.shape, device=DEVICE, generator=forward_generator)
        < flip_probability
    )
    forward_state = forward_state ^ flips.long()
    if time in FORWARD_TIMESTEPS:
        forward_snapshots[time] = forward_state.cpu().numpy().copy()
fig = make_subplots(
    rows=1,
    cols=len(FORWARD_TIMESTEPS) + 1,
    subplot_titles=["Fixed coloring"] + [f"t = {time}" for time in FORWARD_TIMESTEPS],
    horizontal_spacing=0.02,
)
add_board(fig, forward_coloring, forward_coloring, 1, 1)
for col, time in enumerate(FORWARD_TIMESTEPS, start=2):
    add_mask(fig, forward_snapshots[time], forward_coloring, 1, col, queens=time == 0)
style_boards(fig, "D3PM forward diffusion · clean queen mask → binary noise").show(
    config=PLOT_CONFIG
)

## 3. Coloring-conditioned clean-mask model

Reuse the shared U-Net with spatial resolutions
$7 \to 4 \to 2 \to 4 \to 7$, base width 64 and a 128-dimensional time embedding.

- $x_t$: **2 one-hot channels**, one for 0 and one for 1.
- $c$: **7 one-hot channels**, copied unchanged into every network call.
- Row/column coordinates: **2 channels**.
- Concatenated input: $(B,11,7,7)$.
- Output: $(B,2,7,7)$ logits of $\widetilde p_\theta(x_0^i\mid x_t,c,t)$.

The normalized timestep $t/T$ enters the same sinusoidal embedding and residual
blocks as in DFM. Conditioning uses channel concatenation; all spatial layers
can read both the current mask and its region map. The clean target is never
supplied as a condition.

Both binary states are allowed at every pixel. No queen count, row, column or
region constraint is imposed by the model or sampler.


In [7]:
MODEL_CONFIG = dict(
    board_size=BOARD_SIZE,
    base_channels=64,
    time_dim=128,
    time_embedding_scale=100.0,
)


class QueensD3PM(nn.Module):
    def __init__(
        self,
        board_size=7,
        base_channels=64,
        time_dim=128,
        time_embedding_scale=100.0,
    ):
        super().__init__()
        if board_size != BOARD_SIZE:
            raise ValueError("This experiment and dataset use 7×7 boards.")
        self.time_embedding_scale = float(time_embedding_scale)
        self.network = Denoiser(
            embedding_dim=NUM_CLASSES + BOARD_SIZE + 2,
            output_dim=NUM_CLASSES,
            base_channels=base_channels,
            time_dim=time_dim,
        )
        positions = torch.linspace(-1, 1, board_size)
        rr, cc = torch.meshgrid(positions, positions, indexing="ij")
        self.register_buffer("coordinates", torch.stack((rr, cc))[None])

    def forward(self, state, coloring, times):
        state_channels = (
            F.one_hot(state.long(), NUM_CLASSES).permute(0, 3, 1, 2).float()
        )
        color_channels = (
            F.one_hot(coloring.long(), BOARD_SIZE).permute(0, 3, 1, 2).float()
        )
        inputs = torch.cat(
            (
                state_channels,
                color_channels,
                self.coordinates.expand(len(state), -1, -1, -1),
            ),
            dim=1,
        )
        return self.network(inputs, times * self.time_embedding_scale)


model = QueensD3PM(**MODEL_CONFIG).to(DEVICE)
display(
    HTML(
        f"<b>Conditional binary U-Net</b> · "
        f"{sum(p.numel() for p in model.parameters()):,} parameters"
        " · (B, 11, 7, 7) → (B, 2, 7, 7)"
    )
)

## 4. Online D3PM training

For each example independently, draw an augmented puzzle/solution pair,
$t\sim\operatorname{Uniform}\{1,\ldots,T\}$ and fresh binary noise to obtain
$x_t\sim q(x_t\mid x_0)$. One network call predicts clean-mask logits.

The exact per-cell forward posterior is

$$
q(x_{t-1}=k\mid x_t=j,x_0=a)
=\frac{Q_t[k,j]\bar Q_{t-1}[a,k]}{\bar Q_t[a,j]}.
$$

For $t>1$, use the original D3PM clean-state parameterization (Eq. 4):

$$
p_\theta(x_{t-1}=k\mid x_t=j,c)
\propto Q_t[k,j]\sum_{a\in\{0,1\}}
\bar Q_{t-1}[a,k]\widetilde p_\theta(x_0=a\mid x_t,c,t).
$$

Normalize over $k$. At the last reverse step, $t=1$, the decoder is the
predicted clean distribution $\widetilde p_\theta(x_0\mid x_1,c,1)$ directly.
Computation uses log probabilities for numerical stability.

Let $\ell_t$ denote posterior KL for $t>1$, or clean-mask decoder cross-entropy
for $t=1$. Optimize the hybrid objective (Eq. 5):

$$
\mathcal L =
\mathbb E_{t,x_t}\!\left[
  \frac{1}{B\cdot49}\sum_{b,i}
  \left(T\ell_{t_b}^{b,i}
  -\lambda\log\widetilde p_\theta(x_{0,b}^i\mid x_{t_b},c_b,t_b)\right)
\right],\qquad \lambda=0.001.
$$

The factor $T$ makes uniform timestep sampling estimate the sum of variational
terms. Terminal-prior KL is zero because $q(x_T\mid x_0)$ is exactly uniform.
Both classes receive their ordinary pixel contribution; there is no class
weighting or constraint penalty.

The logarithmic loss plot updates in place. Validation uses six fixed
timesteps and reproducible noise. Its hybrid objective is a six-time diagnostic,
not an exact evaluation of the full variational bound. A separate clean-mask
CE curve at $t=T$ measures prediction from pure noise plus coloring.

Run the setup cell before training or loading. To resume, load a checkpoint
and then run the training loop. The training loop adds 100 more epochs each time.


In [8]:
BATCH_SIZE = 64
EPOCHS = 100  # Additional epochs for each execution of the training loop.
SAMPLES_PER_EPOCH = 4096  # Random augmented views, sampled with replacement.
LEARNING_RATE = 2e-4
PLOT_EVERY = 20

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=RandomSampler(
        train_dataset, replacement=True, num_samples=SAMPLES_PER_EPOCH
    ),
    num_workers=0,
    pin_memory=DEVICE.type == "cuda",
)
validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)

AUXILIARY_X0_WEIGHT = 0.001
history = dict(
    step=[],
    train_loss=[],
    validation_step=[],
    validation_loss=[],
    validation_prior_ce=[],
    epochs_completed=0,
)
global_step = 0


def reverse_log_probabilities(clean_logits, state, timesteps):
    # p(k | j) ∝ sum_a q(k,j | a) p_theta(a | j), as in D3PM Eq. 4.
    log_clean = F.log_softmax(clean_logits, dim=1)
    joint_log = diffusion["joint_log"].to(state.device)[timesteps[:, None, None], state]
    unnormalized = torch.logsumexp(
        joint_log + log_clean.permute(0, 2, 3, 1)[..., None], dim=-2
    )
    log_previous = (
        unnormalized - torch.logsumexp(unnormalized, dim=-1, keepdim=True)
    ).permute(0, 3, 1, 2)
    # The t=1 decoder directly predicts x_0.
    return torch.where((timesteps == 1)[:, None, None, None], log_clean, log_previous)


def diffusion_objective(clean_logits, state, clean, timesteps):
    reverse_log = reverse_log_probabilities(clean_logits, state, timesteps)
    true_posterior = (
        diffusion["posterior"]
        .to(state.device)[timesteps[:, None, None], state, clean]
        .permute(0, 3, 1, 2)
    )
    # At t=1 the true posterior is a point mass: KL equals decoder CE.
    per_pixel_vb = F.kl_div(reverse_log, true_posterior, reduction="none").sum(dim=1)
    clean_ce = F.cross_entropy(clean_logits, clean)
    loss = DIFFUSION_STEPS * per_pixel_vb.mean() + AUXILIARY_X0_WEIGHT * clean_ce
    return loss, clean_ce


def training_loss(coloring, target):
    timesteps = torch.randint(1, DIFFUSION_STEPS + 1, (len(coloring),), device=DEVICE)
    state = q_sample(target, timesteps)
    logits = model(state, coloring, timesteps.float() / DIFFUSION_STEPS)
    loss, _ = diffusion_objective(logits, state, target, timesteps)
    return loss


@torch.no_grad()
def validation_losses():
    was_training = model.training
    model.eval()
    generator = torch.Generator(device=DEVICE).manual_seed(SEED + 401)
    total, count, at_prior, count_at_prior = 0.0, 0, 0.0, 0
    # Six network calls per batch, matching the DFM validation budget.
    validation_times = np.rint(np.linspace(1, DIFFUSION_STEPS, 6)).astype(int)
    try:
        for coloring, target in validation_loader:
            coloring, target = coloring.to(DEVICE), target.to(DEVICE)
            for time in validation_times:
                timesteps = torch.full(
                    (len(coloring),), int(time), dtype=torch.long, device=DEVICE
                )
                state = q_sample(target, timesteps, generator)
                logits = model(state, coloring, timesteps.float() / DIFFUSION_STEPS)
                loss, clean_ce = diffusion_objective(logits, state, target, timesteps)
                total += loss.item() * len(coloring)
                count += len(coloring)
                if time == DIFFUSION_STEPS:
                    at_prior += clean_ce.item() * len(coloring)
                    count_at_prior += len(coloring)
        return total / count, at_prior / count_at_prior
    finally:
        model.train(was_training)

In [9]:
training_progress = widgets.IntProgress(
    value=0,
    min=0,
    max=EPOCHS * len(train_loader),
    description="Training",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="100%"),
)
training_status = widgets.HTML()
loss_plot = go.FigureWidget()
loss_plot.add_scatter(
    name="Train hybrid loss · moving mean",
    mode="lines",
    line=dict(color="#4F6BED", width=2),
)
loss_plot.add_scatter(
    name="Validation hybrid loss · fixed noise",
    mode="lines+markers",
    line=dict(color="#E49B43", width=2),
    marker=dict(size=5),
)
loss_plot.add_scatter(
    name="Clean-mask CE at t = T",
    mode="lines+markers",
    line=dict(color="#2F9D88", width=2, dash="dot"),
    marker=dict(size=5),
)
loss_plot.update_layout(
    title="Queens D3PM · variational + clean-mask objective",
    template="plotly_white",
    xaxis_title="Optimizer step",
    yaxis_title="Mean pixel objective · log scale",
    yaxis_type="log",
    autosize=True,
    height=450,
    margin=dict(l=65, r=20, t=65, b=90),
    legend=dict(orientation="h", y=-0.2, x=0),
    font=dict(family="Arial, sans-serif", size=15),
)
display(training_progress, training_status, loss_plot)


def refresh_loss_plot():
    values = np.asarray(history["train_loss"], dtype=float)
    window = min(25, len(values))
    if not window:
        return
    smooth = np.convolve(values, np.ones(window) / window, mode="valid")
    with loss_plot.batch_update():
        loss_plot.data[0].x = history["step"][window - 1 :]
        loss_plot.data[0].y = np.maximum(smooth, 1e-12)
        loss_plot.data[1].x = history["validation_step"]
        loss_plot.data[1].y = np.maximum(history["validation_loss"], 1e-12)
        loss_plot.data[2].x = history["validation_step"]
        loss_plot.data[2].y = np.maximum(history["validation_prior_ce"], 1e-12)


refresh_loss_plot()
start_epoch = int(history["epochs_completed"])
for local_epoch in range(EPOCHS):
    model.train()
    for coloring, target in train_loader:
        coloring, target = coloring.to(DEVICE), target.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        loss = training_loss(coloring, target)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        global_step += 1
        history["step"].append(global_step)
        history["train_loss"].append(float(loss.item()))
        training_progress.value += 1
        if global_step % PLOT_EVERY == 0 or len(history["step"]) == 1:
            refresh_loss_plot()
            training_status.value = (
                f"Epoch {start_epoch + local_epoch + 1} · step {global_step:,} · "
                f"train loss {history['train_loss'][-1]:.5f}"
            )
    val_loss, val_prior_ce = validation_losses()
    history["validation_step"].append(global_step)
    history["validation_loss"].append(val_loss)
    history["validation_prior_ce"].append(val_prior_ce)
    history["epochs_completed"] = start_epoch + local_epoch + 1
    refresh_loss_plot()
    training_status.value = (
        f"Epoch {history['epochs_completed']} · step {global_step:,} · "
        f"validation loss {val_loss:.5f} · t = T clean CE {val_prior_ce:.5f}"
    )
model.eval()
training_progress.bar_style = "success"

IntProgress(value=0, description='Training', layout=Layout(width='100%'), max=6400, style=ProgressStyle(descri…

HTML(value='')

FigureWidget({
    'data': [{'line': {'color': '#4F6BED', 'width': 2},
              'mode': 'lines',
              'name': 'Train hybrid loss · moving mean',
              'type': 'scatter',
              'uid': '4047e8d3-a787-48f8-8dca-40cddb23e891'},
             {'line': {'color': '#E49B43', 'width': 2},
              'marker': {'size': 5},
              'mode': 'lines+markers',
              'name': 'Validation hybrid loss · fixed noise',
              'type': 'scatter',
              'uid': '8197ff09-6472-4480-bec8-1574bc8317a6'},
             {'line': {'color': '#2F9D88', 'dash': 'dot', 'width': 2},
              'marker': {'size': 5},
              'mode': 'lines+markers',
              'name': 'Clean-mask CE at t = T',
              'type': 'scatter',
              'uid': '3d9cce86-95ed-4fa1-8414-10a15b962a36'}],
    'layout': {'autosize': True,
               'font': {'family': 'Arial, sans-serif', 'size': 15},
               'height': 450,
               'legend': {'orientat

## 5. Save and load

### Save

Store model and optimizer weights, loss history, diffusion schedule, auxiliary
weight, training settings, exact dataset fingerprint, split and PyTorch random
state in <code>artifacts/queens7/queens7_d3pm.pt</code>.
DFM checkpoints and evaluation files use separate filenames.


In [10]:
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
torch.save(
    {
        "format_version": "queens7_d3pm_v1",
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "model_config": MODEL_CONFIG,
        "history": history,
        "global_step": global_step,
        "diffusion_config": DIFFUSION_CONFIG,
        "auxiliary_x0_weight": AUXILIARY_X0_WEIGHT,
        "training_config": dict(
            batch_size=BATCH_SIZE,
            samples_per_epoch=SAMPLES_PER_EPOCH,
            learning_rate=LEARNING_RATE,
        ),
        "seed": SEED,
        "dataset_sha256": puzzles["metadata"]["data_sha256"],
        "splits": {
            name: torch.as_tensor(indices.copy()) for name, indices in splits.items()
        },
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_state_all": (
            torch.cuda.get_rng_state_all() if DEVICE.type == "cuda" else []
        ),
    },
    CHECKPOINT_PATH,
)
display(
    HTML(
        f"Saved: <code>{html.escape(str(CHECKPOINT_PATH.relative_to(PROJECT_ROOT)))}</code>"
    )
)

### Load

Run Sections 1–3 and the training setup cell, then skip the training loop and save cell. Load here to continue directly to sampling, or run the training loop to resume. Restore the saved split and reject mismatched dataset files.


In [11]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
if checkpoint["format_version"] != "queens7_d3pm_v1":
    raise ValueError("Unsupported Queens checkpoint format.")
if checkpoint["dataset_sha256"] != puzzles["metadata"]["data_sha256"]:
    raise ValueError("Checkpoint and dataset differ; restore the matching dataset.")
MODEL_CONFIG = checkpoint["model_config"]
DIFFUSION_CONFIG = checkpoint["diffusion_config"]
DIFFUSION_STEPS = int(DIFFUSION_CONFIG["steps"])
GENERATION_STEPS = DIFFUSION_STEPS
diffusion = make_diffusion(DIFFUSION_CONFIG, DEVICE)
AUXILIARY_X0_WEIGHT = float(checkpoint["auxiliary_x0_weight"])
training_config = checkpoint["training_config"]
BATCH_SIZE = int(training_config["batch_size"])
SAMPLES_PER_EPOCH = int(training_config["samples_per_epoch"])
LEARNING_RATE = float(training_config["learning_rate"])
model = QueensD3PM(**MODEL_CONFIG).to(DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
history, global_step = checkpoint["history"], int(checkpoint["global_step"])
SEED = int(checkpoint["seed"])
splits = {name: value.numpy().copy() for name, value in checkpoint["splits"].items()}
train_dataset = QueensDataset(splits["train"], augment=True)
validation_dataset = QueensDataset(splits["validation"])
test_dataset = QueensDataset(splits["test"])
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=RandomSampler(
        train_dataset, replacement=True, num_samples=SAMPLES_PER_EPOCH
    ),
    num_workers=0,
    pin_memory=DEVICE.type == "cuda",
)
validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)
torch.set_rng_state(checkpoint["torch_rng_state"])
if DEVICE.type == "cuda" and checkpoint["cuda_rng_state_all"]:
    if len(checkpoint["cuda_rng_state_all"]) == torch.cuda.device_count():
        torch.cuda.set_rng_state_all(checkpoint["cuda_rng_state_all"])
model.eval()
display(
    HTML(
        f"<b>Loaded Queens D3PM</b> · {history['epochs_completed']} epochs · "
        f"{global_step:,} optimizer steps"
    )
)

## 6. Stochastic reverse-diffusion trajectories

Initialize every cell independently from $\operatorname{Bernoulli}(1/2)$,
independently of the coloring. For $t=T,T-1,\ldots,1$:

1. Predict clean-mask logits from $(x_t,c,t/T)$.
2. Compute the categorical reverse distribution from Section 4.
3. Sample the next binary mask $x_{t-1}$ independently per cell.

There is exactly one network call per step. The coloring is the same at every
call. Unlike the DFM erasure path, reverse D3PM can change either bit in either
direction, correcting earlier choices.

Samples remain binary throughout. The final decoder is sampled too; no final
argmax, forced queen count, backtracking, constraint repair or solution search
is applied. The first panel shows the coloring and the remaining panels show
the binary masks, with the supplied binary answer at the far right.


In [12]:
@torch.no_grad()
def sample_queens(
    candidate, coloring, steps=None, seed=0, capture_steps=(), progress=None
):
    steps = DIFFUSION_STEPS if steps is None else steps
    if not isinstance(steps, int) or steps != DIFFUSION_STEPS:
        raise ValueError(
            f"Use steps={DIFFUSION_STEPS}, matching the trained D3PM schedule."
        )
    device = next(candidate.parameters()).device
    coloring = torch.as_tensor(coloring, dtype=torch.long, device=device)
    if coloring.ndim == 2:
        coloring = coloring[None]
    if coloring.ndim != 3 or tuple(coloring.shape[-2:]) != (BOARD_SIZE, BOARD_SIZE):
        raise ValueError("Expected coloring with shape (B,7,7) or (7,7).")
    if bool(((coloring < 0) | (coloring >= BOARD_SIZE)).any()):
        raise ValueError("Coloring must contain only the seven original region IDs.")
    generator = torch.Generator(device=device).manual_seed(int(seed))
    state = torch.randint(0, 2, coloring.shape, device=device, generator=generator)
    capture_steps = set(map(int, capture_steps))
    if any(step < 0 or step > steps for step in capture_steps):
        raise ValueError("Capture steps must be between zero and steps.")
    snapshots = {0: state.cpu().clone()} if 0 in capture_steps else {}
    was_training = candidate.training
    candidate.eval()
    try:
        for completed in range(1, steps + 1):
            time = steps - completed + 1
            timesteps = torch.full((len(state),), time, dtype=torch.long, device=device)
            clean_logits = candidate(
                state, coloring, timesteps.float() / DIFFUSION_STEPS
            )
            p_queen = reverse_log_probabilities(clean_logits, state, timesteps)[
                :, 1
            ].exp()
            state = (
                torch.rand(state.shape, device=device, generator=generator) < p_queen
            ).long()
            if completed in capture_steps:
                snapshots[completed] = state.cpu().clone()
            if progress is not None:
                progress(completed, steps)
    finally:
        candidate.train(was_training)
    return state.cpu(), snapshots

In [13]:
N_TRAJECTORY_PUZZLES = 3
TRAJECTORY_ATTEMPTS = 2
CAPTURE_STEPS = tuple(
    sorted(set(np.rint(np.linspace(0, GENERATION_STEPS, 6)).astype(int)))
)
trajectory_indices = splits["test"][:N_TRAJECTORY_PUZZLES]
trajectory_progress = widgets.IntProgress(
    value=0,
    max=len(trajectory_indices) * TRAJECTORY_ATTEMPTS * GENERATION_STEPS,
    description="Trajectories",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="100%"),
)
display(trajectory_progress)
trajectory_records = []
for puzzle_index in trajectory_indices:
    for attempt in range(TRAJECTORY_ATTEMPTS):
        offset = len(trajectory_records) * GENERATION_STEPS

        def update_trajectory(done, total, offset=offset):
            trajectory_progress.value = offset + done

        final, snapshots = sample_queens(
            model,
            regions[puzzle_index],
            steps=GENERATION_STEPS,
            seed=SEED + 1000 + len(trajectory_records),
            capture_steps=CAPTURE_STEPS,
            progress=update_trajectory,
        )
        solved = bool(
            solution_checks(
                regions[puzzle_index],
                mask_to_tokens(final[0].numpy(), regions[puzzle_index]),
            )["solved"]
        )
        trajectory_records.append((puzzle_index, attempt, snapshots, solved))
trajectory_progress.bar_style = "success"

for puzzle_index, attempt, snapshots, solved in trajectory_records:
    titles = (
        ["Fixed coloring"]
        + [f"t = {GENERATION_STEPS - step}" for step in CAPTURE_STEPS]
        + ["Supplied mask"]
    )
    fig = make_subplots(
        rows=1, cols=len(titles), subplot_titles=titles, horizontal_spacing=0.018
    )
    add_board(fig, regions[puzzle_index], regions[puzzle_index], 1, 1)
    for col, step in enumerate(CAPTURE_STEPS, start=2):
        add_mask(
            fig,
            snapshots[step][0].numpy(),
            regions[puzzle_index],
            1,
            col,
            queens=step == GENERATION_STEPS,
        )
    add_mask(
        fig,
        queen_masks[puzzle_index],
        regions[puzzle_index],
        1,
        len(titles),
        queens=True,
    )
    source = puzzles["source_names"][puzzle_index]
    status = "SOLVED" if solved else "UNSOLVED"
    style_boards(
        fig,
        f"D3PM · {source} · test puzzle {puzzle_index} · attempt {attempt + 1} · {status}",
    ).show(config=PLOT_CONFIG)

IntProgress(value=0, description='Trajectories', layout=Layout(width='100%'), max=600, style=ProgressStyle(des…

## 7. How often is a held-out puzzle solved?

Run independent stochastic trajectories from independent binary noise, conditioned on each selected test puzzle's coloring.

- **Solve probability:** average success over puzzles and attempts.
- **Solved at least once:** fraction of puzzles solved by any of the repeated attempts.
- **Exact match:** all output tokens equal the supplied target.
- Strict scoring checks every Queens rule on the raw output. Checking a proposed answer is linear-time; it does not search for solutions or verify uniqueness.

The 95% interval bootstraps whole puzzles to preserve the grouping of repeated attempts. Source-specific rates separate externally collected and synthetic puzzles.

The retry curve shows the probability of finding at least one solution within k attempts. It averages over all subsets of k recorded attempts for each puzzle, then across puzzles. The shaded band is a 95% puzzle-bootstrap interval. The curve stops at the number of attempts actually sampled.

The orange dashed reference assumes the same single-attempt probability $\bar p$ for every puzzle and shows $1-(1-\bar p)^k$. The blue curve estimates success separately for each puzzle before averaging.


    **Paired comparison with DFM.** The default seed, split, permutation of test
    puzzles, number of attempts, evaluation batch size and neural-call budget match
    <code>01_queens_discrete_flow_matching.ipynb</code>. The noise and random choices
    are method-specific; compare the same puzzle IDs. Keep dataset SHA and actual
    training-step count equal when comparing saved evaluation files.


In [14]:
N_EVAL_PUZZLES = 256  # None evaluates every held-out puzzle.
ATTEMPTS_PER_PUZZLE = 32
EVAL_BATCH_SIZE = 128
EVAL_SEED = SEED + 10_000
if ATTEMPTS_PER_PUZZLE < 1 or EVAL_BATCH_SIZE < 1:
    raise ValueError("Attempt count and batch size must be positive.")
evaluation_rng = np.random.default_rng(EVAL_SEED)
evaluation_indices = evaluation_rng.permutation(splits["test"])
if N_EVAL_PUZZLES is not None:
    if N_EVAL_PUZZLES < 1:
        raise ValueError("N_EVAL_PUZZLES must be positive or None.")
    evaluation_indices = evaluation_indices[:N_EVAL_PUZZLES]
flat_indices = np.repeat(evaluation_indices, ATTEMPTS_PER_PUZZLE)
evaluation_progress = widgets.IntProgress(
    value=0,
    max=len(flat_indices) * GENERATION_STEPS,
    description="Evaluation",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="100%"),
)
evaluation_status = widgets.HTML()
display(evaluation_progress, evaluation_status)
predicted_batches = []
for start in range(0, len(flat_indices), EVAL_BATCH_SIZE):
    batch_indices = flat_indices[start : start + EVAL_BATCH_SIZE]
    offset = start * GENERATION_STEPS

    def update_evaluation(done, total, offset=offset, count=len(batch_indices)):
        evaluation_progress.value = offset + done * count

    prediction, _ = sample_queens(
        model,
        regions[batch_indices],
        steps=GENERATION_STEPS,
        seed=EVAL_SEED + start,
        progress=update_evaluation,
    )
    predicted_batches.append(mask_to_tokens(prediction.numpy(), regions[batch_indices]))
    evaluation_status.value = (
        f"{start + len(batch_indices):,} / {len(flat_indices):,} trajectories"
    )
evaluation_progress.bar_style = "success"

predicted_tokens = np.concatenate(predicted_batches, axis=0)
checks = solution_checks(regions[flat_indices], predicted_tokens)
successes = checks["solved"].reshape(len(evaluation_indices), ATTEMPTS_PER_PUZZLE)
exact_matches = np.all(
    predicted_tokens == target_tokens[flat_indices], axis=(-2, -1)
).reshape(successes.shape)
per_puzzle_success = successes.mean(axis=1)
solve_probability = float(per_puzzle_success.mean())
bootstrap_rng = np.random.default_rng(EVAL_SEED + 1)
bootstrap_means = per_puzzle_success[
    bootstrap_rng.integers(
        0, len(per_puzzle_success), size=(3000, len(per_puzzle_success))
    )
].mean(axis=1)
lower, upper = np.quantile(bootstrap_means, [0.025, 0.975])
at_least_once = float(successes.any(axis=1).mean())
display(
    HTML(
        f"<b>Solved: {100 * solve_probability:.2f}%</b>"
        f" · 95% puzzle-bootstrap interval [{100 * lower:.2f}%, {100 * upper:.2f}%]<br>"
        f"Exact target match: {100 * exact_matches.mean():.2f}% · "
        f"Solved at least once in {ATTEMPTS_PER_PUZZLE} attempts: {100 * at_least_once:.2f}%<br>"
        f"{len(evaluation_indices)} unseen puzzles · {len(flat_indices):,} trajectories · "
        f"{GENERATION_STEPS} steps · mean queen count {checks['queen_count'].mean():.2f}"
    )
)
evaluation_results = dict(
    dataset_indices=evaluation_indices,
    predicted_tokens=predicted_tokens.reshape(
        len(evaluation_indices), ATTEMPTS_PER_PUZZLE, BOARD_SIZE, BOARD_SIZE
    ),
    successes=successes,
    exact_matches=exact_matches,
    per_puzzle_success=per_puzzle_success,
    steps=GENERATION_STEPS,
    predicted_masks=(predicted_tokens != EMPTY).reshape(
        len(evaluation_indices), ATTEMPTS_PER_PUZZLE, BOARD_SIZE, BOARD_SIZE
    ),
    method="d3pm",
    training_optimizer_steps=global_step,
    training_epochs=int(history["epochs_completed"]),
    model_parameter_count=sum(p.numel() for p in model.parameters()),
    batch_size=BATCH_SIZE,
    samples_per_epoch=SAMPLES_PER_EPOCH,
    seed=SEED,
)

IntProgress(value=0, description='Evaluation', layout=Layout(width='100%'), max=819200, style=ProgressStyle(de…

HTML(value='')

In [15]:
source_names = puzzles["source_names"][evaluation_indices]
names = ["All test puzzles"] + sorted(set(source_names.tolist()))
means, lows, highs, sample_counts = [], [], [], []
for name in names:
    values = (
        per_puzzle_success
        if name == "All test puzzles"
        else per_puzzle_success[source_names == name]
    )
    draw = bootstrap_rng.integers(0, len(values), size=(3000, len(values)))
    interval = np.quantile(values[draw].mean(axis=1), [0.025, 0.975])
    means.append(100 * float(values.mean()))
    lows.append(100 * float(interval[0]))
    highs.append(100 * float(interval[1]))
    sample_counts.append(len(values))

# For a puzzle with c successes in n recorded attempts, average over all
# subsets of k attempts: pass@k = 1 - C(n-c, k) / C(n, k).
# This uses the actual observed attempts and does not extrapolate beyond n.
attempt_count = successes.shape[1]
attempt_counts = np.arange(attempt_count + 1)
per_puzzle_pass_at_k = np.zeros((len(successes), attempt_count + 1))
no_solution_probability = np.ones(len(successes))
failed_attempts = attempt_count - successes.sum(axis=1)
for k in range(1, attempt_count + 1):
    no_solution_probability *= np.maximum(failed_attempts - k + 1, 0) / (
        attempt_count - k + 1
    )
    per_puzzle_pass_at_k[:, k] = 1.0 - no_solution_probability
pass_at_k = per_puzzle_pass_at_k.mean(axis=0)
mean_single_attempt = float(successes.mean())
mean_probability_pass_at_k = 1.0 - (1.0 - mean_single_attempt) ** attempt_counts

# Resample whole puzzles, keeping their repeated attempts together.
# Bootstrap weights avoid allocating a (bootstrap, puzzles, attempts) array.
retry_rng = np.random.default_rng(EVAL_SEED + 2)
retry_bootstrap_weights = retry_rng.multinomial(
    len(successes),
    np.full(len(successes), 1.0 / len(successes)),
    size=3000,
)
retry_bootstrap = retry_bootstrap_weights @ per_puzzle_pass_at_k / len(successes)
pass_at_k_lower, pass_at_k_upper = np.quantile(
    retry_bootstrap,
    [0.025, 0.975],
    axis=0,
)

fig = make_subplots(
    rows=1,
    cols=2,
    horizontal_spacing=0.12,
    subplot_titles=[
        "Solve probability - 95% intervals",
        "At least one solution vs number of attempts",
    ],
)
fig.add_trace(
    go.Bar(
        x=[f"{name}<br>n = {count}" for name, count in zip(names, sample_counts)],
        y=means,
        marker_color=["#4F6BED"] + ["#6CAEA0"] * (len(names) - 1),
        error_y=dict(
            type="data",
            symmetric=False,
            array=np.maximum(0, np.asarray(highs) - means),
            arrayminus=np.maximum(0, np.asarray(means) - lows),
        ),
        hovertemplate="%{x}<br>solved %{y:.2f}%<extra></extra>",
        showlegend=False,
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=attempt_counts,
        y=100 * pass_at_k_lower,
        mode="lines",
        line=dict(width=0),
        hoverinfo="skip",
        showlegend=False,
    ),
    row=1,
    col=2,
)
fig.add_trace(
    go.Scatter(
        x=attempt_counts,
        y=100 * pass_at_k_upper,
        mode="lines",
        line=dict(width=0),
        fill="tonexty",
        fillcolor="rgba(79,107,237,0.16)",
        hoverinfo="skip",
        showlegend=False,
    ),
    row=1,
    col=2,
)
fig.add_trace(
    go.Scatter(
        x=attempt_counts,
        y=100 * pass_at_k,
        mode="lines+markers",
        line=dict(color="#4F6BED", width=3),
        marker=dict(size=7),
        customdata=np.column_stack((100 * pass_at_k_lower, 100 * pass_at_k_upper)),
        hovertemplate=(
            "Attempts: %{x}<br>At least one solution: %{y:.2f}%"
            "<br>95% interval: [%{customdata[0]:.2f}%, %{customdata[1]:.2f}%]"
            "<extra></extra>"
        ),
        name="Per-puzzle estimate",
        showlegend=True,
    ),
    row=1,
    col=2,
)
fig.add_trace(
    go.Scatter(
        x=attempt_counts,
        y=100 * mean_probability_pass_at_k,
        mode="lines+markers",
        name="From mean success probability",
        line=dict(color="#D88437", width=3, dash="dash"),
        marker=dict(size=6),
        hovertemplate=(
            "Attempts: %{x}<br>From mean probability: %{y:.2f}%" "<extra></extra>"
        ),
        showlegend=True,
    ),
    row=1,
    col=2,
)
fig.update_yaxes(title_text="Solved trajectories (%)", range=[0, 105], row=1, col=1)
fig.update_yaxes(
    title_text="Probability of at least one solution (%)",
    rangemode="tozero",
    row=1,
    col=2,
)
fig.update_xaxes(
    title_text="Number of attempts",
    dtick=max(1, attempt_count // 10),
    range=[0, attempt_count],
    row=1,
    col=2,
)
fig.update_layout(
    title=dict(text=f"Queens D3PM - held-out results - {GENERATION_STEPS} steps", x=0),
    template="plotly_white",
    autosize=True,
    height=520,
    legend=dict(orientation="h", x=0.5, xanchor="center", y=-0.28, yanchor="top"),
    bargap=0.2,
    margin=dict(l=65, r=20, t=100, b=140),
    font=dict(family="Arial, sans-serif", size=16),
)
fig.show(config=PLOT_CONFIG)
evaluation_results.update(
    mean_single_attempt=mean_single_attempt,
    mean_probability_pass_at_k=mean_probability_pass_at_k,
    attempt_counts=attempt_counts,
    pass_at_k=pass_at_k,
    pass_at_k_lower=pass_at_k_lower,
    pass_at_k_upper=pass_at_k_upper,
)
np.savez_compressed(
    ARTIFACTS / "queens7_d3pm_evaluation.npz",
    **evaluation_results,
    dataset_sha256=puzzles["metadata"]["data_sha256"],
    source_names=source_names,
    evaluation_seed=EVAL_SEED,
)

### 7.1 From retry success to puzzle difficulty

The orange dashed curve uses the mean single-attempt success probability $\bar p$:

$$
P_{\mathrm{mean}}(k)=1-(1-\bar p)^k.
$$

This assumes every puzzle has the same success probability. For puzzles with individual probabilities $p_i$, independent retries on the same puzzle instead give

$$
P_{\mathrm{puzzles}}(k)
=\mathbb E_i\!\left[1-(1-p_i)^k\right].
$$

For $k\ge2$, the function $1-(1-p)^k$ is concave, so

$$
\mathbb E_i\!\left[1-(1-p_i)^k\right]
\le 1-\left(1-\mathbb E_i[p_i]\right)^k.
$$

Some puzzles are already relatively easy and are often solved repeatedly; others remain difficult across retries. Pooling them into one mean success probability overestimates the benefit of repeated attempts when their probabilities differ.

A gap between the orange and blue curves therefore motivates a study of **different puzzle difficulties for this model and sampler**. We estimate those differences from 32 independent attempts per puzzle; finite sampling also contributes uncertainty.


### 7.2 Solvability study

Estimate each held-out puzzle's solvability as its fraction of successful attempts:

$$
\widehat p_i=\frac{\text{successful attempts for puzzle }i}{32}.
$$

Plot a continuous, boundary-corrected kernel density of these estimates. The x-axis is **probability to be solved**; the y-axis is **% of puzzles on a logarithmic scale**, expressed per one percentage point of x so that the area under the density equals 100%.

The smoothing bandwidth defaults to one attempt's resolution ($100/32=3.125$ percentage points). This smooths the measured probabilities; it does not increase the information in 32 attempts. Zero observed successes means no successful attempt was observed for that puzzle. Very small density tails below 0.001% per percentage point are hidden on the log plot.

Rerun the evaluation cell with 32 attempts before this study. The three marked probabilities correspond to the example puzzles below.


In [16]:
study_successes = np.asarray(successes, dtype=bool)
study_attempts = study_successes.shape[1]
if study_attempts < 32:
    raise ValueError(
        "Solvability study needs at least 32 recorded attempts per puzzle. "
        "Rerun the evaluation cell with ATTEMPTS_PER_PUZZLE = 32."
    )
study_rates = study_successes.mean(axis=1)
study_probabilities_percent = 100.0 * study_rates

SOLVABILITY_BANDWIDTH_PP = 100.0 / study_attempts
SOLVABILITY_DENSITY_FLOOR = 0.001
if SOLVABILITY_BANDWIDTH_PP <= 0 or SOLVABILITY_DENSITY_FLOOR <= 0:
    raise ValueError("Bandwidth and the log-display density floor must be positive.")
solvability_grid = np.linspace(0.0, 100.0, 2001)
bandwidth = float(SOLVABILITY_BANDWIDTH_PP)
distance = (
    solvability_grid[:, None] - study_probabilities_percent[None, :]
) / bandwidth
left_reflection = (
    solvability_grid[:, None] + study_probabilities_percent[None, :]
) / bandwidth
right_reflection = (
    solvability_grid[:, None] - (200.0 - study_probabilities_percent[None, :])
) / bandwidth
solvability_density = (
    np.exp(-0.5 * distance**2)
    + np.exp(-0.5 * left_reflection**2)
    + np.exp(-0.5 * right_reflection**2)
).mean(axis=1) / (bandwidth * np.sqrt(2.0 * np.pi))
# Reflection retains boundary mass at 0% and 100%; normalize the sampled grid.
density_area = np.sum(
    0.5
    * (solvability_density[:-1] + solvability_density[1:])
    * np.diff(solvability_grid)
)
solvability_density_percent = 100.0 * solvability_density / density_area
visible_density = np.where(
    solvability_density_percent >= SOLVABILITY_DENSITY_FLOOR,
    solvability_density_percent,
    np.nan,
)

# Pick distinct examples: highest observed rate, then approximately half
# the previous selected example's rate. Use actual available measurements.
study_example_positions = []
study_example_targets = []
remaining_positions = np.arange(len(study_rates))
previous_rate = None
for _ in range(min(3, len(study_rates))):
    if previous_rate is None:
        target_rate = float(study_rates.max())
        candidates = remaining_positions
    else:
        target_rate = previous_rate / 2.0
        lower_candidates = remaining_positions[
            study_rates[remaining_positions] < previous_rate
        ]
        candidates = lower_candidates if len(lower_candidates) else remaining_positions
    differences = np.abs(study_rates[candidates] - target_rate)
    position = int(candidates[np.argmin(differences)])
    study_example_positions.append(position)
    study_example_targets.append(target_rate)
    previous_rate = float(study_rates[position])
    remaining_positions = remaining_positions[remaining_positions != position]

study_example_colors = ["#D88437", "#2F9D88", "#9B6BD3"]
solvability_fig = go.Figure(
    go.Scatter(
        x=solvability_grid,
        y=visible_density,
        mode="lines",
        line=dict(color="#4F6BED", width=3),
        name="Smoothed measured solvability",
        showlegend=False,
        hovertemplate=(
            "Probability to be solved: %{x:.2f}%"
            "<br>Puzzles per 1 percentage point: %{y:.4f}%<extra></extra>"
        ),
    )
)
for rank, position in enumerate(study_example_positions):
    probability_percent = float(study_probabilities_percent[position])
    density_at_probability = float(
        np.interp(probability_percent, solvability_grid, solvability_density_percent)
    )
    solvability_fig.add_vline(
        x=probability_percent,
        line=dict(color=study_example_colors[rank], width=1.3, dash="dot"),
    )
    solvability_fig.add_trace(
        go.Scatter(
            x=[probability_percent],
            y=[density_at_probability],
            mode="markers",
            marker=dict(color=study_example_colors[rank], size=10),
            name=f"Example {rank + 1}: {probability_percent:.2f}%",
            hovertemplate=(
                f"Example {rank + 1}<br>Measured solvability: {probability_percent:.2f}%"
                "<extra></extra>"
            ),
        )
    )
solvability_fig.update_layout(
    title=dict(
        text=f"D3PM solvability study - {len(study_rates)} puzzles, "
        f"{study_attempts} attempts each",
        x=0,
    ),
    template="plotly_white",
    autosize=True,
    height=500,
    xaxis_title="Probability to be solved (%)",
    yaxis_title="% of puzzles per 1 percentage point (log scale)",
    yaxis_type="log",
    legend=dict(orientation="h", x=0, y=-0.24),
    margin=dict(l=85, r=25, t=85, b=115),
    font=dict(family="Arial, sans-serif", size=16),
)
solvability_fig.update_xaxes(range=[0, 100], dtick=10)
solvability_fig.update_yaxes(
    range=[
        np.log10(SOLVABILITY_DENSITY_FLOOR),
        np.ceil(np.log10(solvability_density_percent.max())) + 0.1,
    ]
)
solvability_fig.show(config=PLOT_CONFIG)
display(
    HTML(
        f"Measured resolution: {100 / study_attempts:.3f} percentage points per success."
        f" Smoothing bandwidth: {bandwidth:.3f} percentage points.<br>"
        f"No observed success: {np.mean(study_rates == 0):.1%} of puzzles."
    )
)

np.savez_compressed(
    ARTIFACTS / "queens7_d3pm_solvability_study.npz",
    dataset_indices=evaluation_indices,
    successes=study_successes,
    per_puzzle_probability=study_rates,
    grid_probability_percent=solvability_grid,
    density_percent_per_percentage_point=solvability_density_percent,
    bandwidth_percentage_points=bandwidth,
    example_positions=np.asarray(study_example_positions),
    dataset_sha256=puzzles["metadata"]["data_sha256"],
)

### 7.3 Example puzzles by solvability

Start with the most frequently solved evaluated puzzle. Then select a different puzzle whose measured success probability is closest to half of the previous example's probability, preferring a strictly lower rate when one is available.

The examples appear in decreasing solvability order. Titles show the **actual measured probability and success count**, since exact 100%, 50% and 25% examples may not exist. The queens shown are the supplied ground-truth solution on the original coloring.


In [17]:
example_titles = []
for rank, position in enumerate(study_example_positions):
    count = int(study_successes[position].sum())
    probability_percent = float(study_probabilities_percent[position])
    example_titles.append(
        f"Example {rank + 1}: {probability_percent:.2f}%"
        f"<br>{count}/{study_attempts} successful attempts"
    )
study_examples_fig = make_subplots(
    rows=1,
    cols=len(study_example_positions),
    subplot_titles=example_titles,
    horizontal_spacing=0.065,
)
selected_study_examples = []
for col, position in enumerate(study_example_positions, start=1):
    puzzle_index = int(evaluation_indices[position])
    add_board(study_examples_fig, regions[puzzle_index], regions[puzzle_index], 1, col)
    queen_rows, queen_cols = np.where(queen_masks[puzzle_index])
    study_examples_fig.add_trace(
        go.Scatter(
            x=queen_cols,
            y=queen_rows,
            mode="text",
            text=[chr(0x265B)] * len(queen_rows),
            textfont=dict(size=25, color="#172033"),
            hoverinfo="skip",
            showlegend=False,
        ),
        row=1,
        col=col,
    )
    selected_study_examples.append(
        {
            "dataset_index": puzzle_index,
            "id": puzzles["source_ids"][puzzle_index],
            "source": puzzles["source_names"][puzzle_index],
            "successes": int(study_successes[position].sum()),
            "attempts": study_attempts,
            "probability": float(study_rates[position]),
            "target_probability": float(study_example_targets[col - 1]),
        }
    )
style_boards(
    study_examples_fig,
    "Solvability study - example puzzles",
    rows=1,
)
study_examples_fig.update_layout(height=410, margin=dict(l=10, r=10, t=120, b=20))
study_examples_fig.show(config=PLOT_CONFIG)
display(
    HTML(
        "<br>".join(
            f"<b>Example {rank + 1}</b>: "
            f"{html.escape(str(example['source']))} / {html.escape(str(example['id']))}"
            for rank, example in enumerate(selected_study_examples)
        )
    )
)